This builds the bar chart comparing coverage between VAMP-seq and SGE assays

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from matplotlib.patches import Polygon

In [ ]:
#Paths to each of the SGE and VAMP-seq subsets from the main dataframe
sge_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251202_SGEsubset.xlsx')
vampseq_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251204_VAMPseqsubset_wDups.xlsx')

In [ ]:
def read_data(sge, vampseq): #Reads all data and changes variant consequence annotations to be more human readable
 
    sge_genes = ['BARD1', 'PALB2', 'BRCA2', 'RAD51D', 'XRCC2', 'CTCF', 'SFPQ'] #SGE genes

    vampseq_genes = ['G6PD', 'TSC2', 'F9'] #VAMP-seq genes
    
    sge_dict = {} #Empty dictionary to store SGE data in form or {gene_name: gene_data}

    for gene in sge_genes:
        print(f' Reading {gene}...')
        df = sge.loc[sge['Gene'] == gene]
        #print(df)
        df = df.loc[df['ref_allele'].str.len() == 1]
        df = df.dropna(subset = ['simplified_consequence'])
        df = df.rename(columns = {'hg38_start': 'pos'})
        df['amino_acid_change'] = df['aa_ref'] + df['aa_pos'].astype(str) + df['aa_alt']
        
        sge_dict[gene] = df


    #Analagous code for reading VAMP-seq data
    vamp_dict = {}
    for gene in vampseq_genes:
        print(f' Reading {gene}...')
        
        #Merges TSC2 datasets into single dataframe
        if gene == 'TSC2': 
            df = vampseq.loc[vampseq['Gene'] == gene]

            rapgap_df = df.loc[df['Dataset'].str.contains('rapgap')].copy()
            tuberin_df = df.loc[df['Dataset'].str.contains('tuberin')].copy()
            tsc2_sets = [('TSC2 RapGAP', rapgap_df), ('TSC2 Tuberin', tuberin_df)]

            for set in tsc2_sets:
                name, df = set
                df = df.drop_duplicates(subset = ['ID'])
                df = df.rename(columns = {'hg38_start': 'pos'})
                df['amino_acid_change'] = df['aa_ref'] + df['aa_pos'].astype(str) + df['aa_alt']
                vamp_dict[name] = df
        else:
            df = vampseq.loc[vampseq['Gene'] == gene]

            df = df.drop_duplicates(subset = ['ID'])
            df = df.rename(columns = {'hg38_start': 'pos'})
            df['amino_acid_change'] = df['aa_ref'] + df['aa_pos'].astype(str) + df['aa_alt']
        
            vamp_dict[gene] = df

    #print(sge_dict)
    return sge_dict, vamp_dict

In [ ]:
def assay_comparison_bars(sge_data, vampseq_data):
    # Calculate totals   
    sge_total_genomic = sum([len(set(sge_data[g]['pos'])) for g in sge_data])

    sge_total_amino = sum([sge_data[g]['amino_acid_change'].nunique() for g in sge_data])
    
    vamp_total_genomic = 0
    vamp_total_amino = 0
    for gene in vampseq_data:
        df = vampseq_data[gene]
        vamp_total_genomic += len(set(df['aa_pos'])) * 3
        vamp_total_amino += len(set(df['amino_acid_change']))
    
    # Data for stacked bars 
    categories = ['Genomic positions', 'Amino acid changes']
    sge_values = [sge_total_genomic, sge_total_amino]
    vamp_values = [vamp_total_genomic, vamp_total_amino]
    
    fig, ax = plt.subplots(figsize=(8, 3))
    
    # Create horizontal stacked bars
    y = np.arange(len(categories))
    height = 0.6
    
    p1 = ax.barh(y, vamp_values, height, label='VAMP-seq', color='#F4A261', alpha=0.9)
    p2 = ax.barh(y, sge_values, height, left=vamp_values, label='SGE', color='#A8DADC', alpha=0.9)
    
    # Add connecting polygons - VAMP-seq sections 
    vamp_genomic_left = vamp_values[0]
    vamp_genomic_right = vamp_values[1]
    
    poly_vamp = Polygon([
        (0, y[0] + height/2),
        (vamp_genomic_left, y[0] + height/2),
        (vamp_genomic_right, y[1] - height/2),
        (0, y[1] - height/2)
    ], facecolor='#F4A261', alpha=0.3, edgecolor='none')
    ax.add_patch(poly_vamp)
    
    # SGE sections
    sge_genomic_bottom_left = vamp_values[0]
    sge_genomic_top_left = vamp_values[0] + sge_values[0]
    sge_amino_bottom_right = vamp_values[1]
    sge_amino_top_right = vamp_values[1] + sge_values[1]
    
    poly_sge = Polygon([
        (sge_genomic_bottom_left, y[0] + height/2),
        (sge_genomic_top_left, y[0] + height/2),
        (sge_amino_top_right, y[1] - height/2),
        (sge_amino_bottom_right, y[1] - height/2)
    ], facecolor='#A8DADC', alpha=0.3, edgecolor='none')
    ax.add_patch(poly_sge)
    
    # Formatting
    ax.set_xlabel('Count', fontsize=14, fontweight='bold')
    ax.set_title('Assay Coverage Comparison', fontsize=16, fontweight='bold', pad=20)
    ax.set_yticks(y)
    ax.set_yticklabels(categories, fontsize=12, fontweight='bold')
    ax.legend(fontsize=12, frameon=False, loc='lower right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Reduce space between bars
    ax.set_ylim(-0.5, len(categories) - 0.5)
    
    # Add value labels on bars
    for i, (v, s) in enumerate(zip(vamp_values, sge_values)):
        ax.text(v/2, i, f'{v:,}', ha='center', va='center', fontsize=10, fontweight='bold', color='black')
        ax.text(v + s/2, i, f'{s:,}', ha='center', va='center', fontsize=10, fontweight='bold', color='black')
    
    plt.tight_layout()
    plt.show()
    #fig.savefig('/Users/ivan/Desktop/pillar_project_figs/20251229_vampseq_sge_bars.svg')

In [ ]:
def main():
    sge_data, vampseq_data = read_data(sge_ppj_subset, vampseq_ppj_subset)

    assay_comparison_bars(sge_data, vampseq_data)

In [ ]:
main()